# 09 — Classifier Two-Sample Test (XGBoost)
ECOD ceilings at 95% because it assumes feature independence — it sums per-feature tail probabilities and dilutes anomalies that are extreme only in *joint* structure. CTST attacks the joint structure directly.

**Setup:** binary classifier on `fit_pool = train ∪ val` (label 0, confirmed-normal) vs `test` (label 1, mostly-normal + injected anomalies). Most test rows are indistinguishable from fit, so the classifier's loss minimum is reached by ranking the *systematic* difference — the anomalies — at the top.

Theoretical posterior at convergence: P(class=1 | x) ≈ 1.0 for anomalous x (only appears in test), ≈ test_prior ≈ 0.15 for normal x. AUC-ranks reflect this.

Writes `submission_ctst.csv`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
import xgboost as xgb
from src.features import build_features, build_temporal_features, feature_names_all

cfg = yaml.safe_load((ROOT / 'configs' / 'config.yaml').read_text())
cols = feature_names_all(cfg['features']['raw'])
SEED = cfg['seed']

In [2]:
proc = ROOT / 'data' / 'processed'
raw = ROOT / cfg['paths']['raw_dir']

train = pd.read_parquet(proc / 'train.parquet')
val_raw = pd.read_csv(raw / cfg['data']['val_file'])
val_raw[cfg['features']['raw']] = val_raw[cfg['features']['raw']].fillna(0.0)
val = build_features(val_raw)

fit_df = build_temporal_features(pd.concat([train, val], ignore_index=True))
test = build_temporal_features(pd.read_parquet(proc / 'test.parquet'))

def mat(df):
    X = df[cols].to_numpy(dtype=np.float32)
    np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return X

X_fit, X_te = mat(fit_df), mat(test)
print('fit:', X_fit.shape, 'test:', X_te.shape)

fit: (1537961, 54) test: (289101, 54)


In [3]:
X = np.concatenate([X_fit, X_te], axis=0)
y = np.concatenate([np.zeros(len(X_fit), dtype=np.int8), np.ones(len(X_te), dtype=np.int8)])

rng = np.random.default_rng(SEED)
idx = rng.permutation(len(X))
split = int(0.9 * len(X))
tr_idx, val_idx = idx[:split], idx[split:]
scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
print('scale_pos_weight:', round(float(scale_pos_weight), 3))

scale_pos_weight: 5.32


In [4]:
clf = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=50,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=float(scale_pos_weight),
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    n_jobs=-1,
    random_state=SEED,
    early_stopping_rounds=20,
)
clf.fit(X[tr_idx], y[tr_idx], eval_set=[(X[val_idx], y[val_idx])], verbose=False)
print('best_iter:', clf.best_iteration, 'best_AUC:', round(float(clf.best_score), 5))

best_iter: 109 best_AUC: 1.0


In [5]:
scores = clf.predict_proba(X_te)[:, 1]
print('min/max/mean:', round(float(scores.min()), 4), round(float(scores.max()), 4), round(float(scores.mean()), 4))

min/max/mean: 0.0195 0.9986 0.9975


In [6]:
out = ROOT / 'outputs' / 'submissions'
pd.DataFrame({'ID': test['index'], 'TARGET': scores}).to_csv(out / 'submission_ctst.csv', index=False)